In [5]:
import os
import pandas as pd
import numpy as np

# Move to our project folder
os.chdir(r"D:\Projects\berlin-electricity-temperature")

print("Ready!")
print("Current folder:", os.getcwd())

Ready!
Current folder: D:\Projects\berlin-electricity-temperature


In [8]:
# Load first consumption file and see what it looks like
df1 = pd.read_csv("data/raw/smard_electricity/smard_2015_2019.csv", sep=None, engine='python')

print("Shape:", df1.shape)
print("Columns:", list(df1.columns))
print(df1.head(3))

Shape: (43824, 6)
Columns: ['\ufeffStart date', 'End date', 'grid load [MWh] Calculated resolutions', 'Grid load incl. hydro pumped storage [MWh] Calculated resolutions', 'Hydro pumped storage [MWh] Calculated resolutions', 'Residual load [MWh] Calculated resolutions']
            ﻿Start date             End date  \
0  Jan 1, 2015 12:00 AM  Jan 1, 2015 1:00 AM   
1   Jan 1, 2015 1:00 AM  Jan 1, 2015 2:00 AM   
2   Jan 1, 2015 2:00 AM  Jan 1, 2015 3:00 AM   

  grid load [MWh] Calculated resolutions  \
0                              44,600.25   
1                              43,454.75   
2                              41,963.25   

  Grid load incl. hydro pumped storage [MWh] Calculated resolutions  \
0                                          45,201.75                  
1                                          43,801.00                  
2                                          42,485.00                  

  Hydro pumped storage [MWh] Calculated resolutions  \
0                   

In [10]:
# Print exact column names to see hidden characters
for col in df1.columns:
    print(repr(col))

'\ufeffStart date'
'End date'
'grid load [MWh] Calculated resolutions'
'Grid load incl. hydro pumped storage [MWh] Calculated resolutions'
'Hydro pumped storage [MWh] Calculated resolutions'
'Residual load [MWh] Calculated resolutions'


In [11]:
# Read file with utf-8-sig encoding which automatically removes the \ufeff character
df1 = pd.read_csv(
    "data/raw/smard_electricity/smard_2015_2019.csv",
    sep=None,
    engine='python',
    encoding='utf-8-sig'    # this removes the hidden \ufeff automatically
)

# Now keep only what we need
df1 = df1[['Start date', 'grid load [MWh] Calculated resolutions']]
df1.columns = ['datetime', 'demand_mwh']

print("Shape:", df1.shape)
print(df1.head(3))

Shape: (43824, 2)
               datetime demand_mwh
0  Jan 1, 2015 12:00 AM  44,600.25
1   Jan 1, 2015 1:00 AM  43,454.75
2   Jan 1, 2015 2:00 AM  41,963.25


In [12]:
# Load second consumption file the same way
df2 = pd.read_csv(
    "data/raw/smard_electricity/smard_2020_2026.csv",
    sep=None,
    engine='python',
    encoding='utf-8-sig'
)

# Keep only what we need
df2 = df2[['Start date', 'grid load [MWh] Calculated resolutions']]
df2.columns = ['datetime', 'demand_mwh']

print("Shape:", df2.shape)
print(df2.head(3))
print(df2.tail(3))

Shape: (54000, 2)
               datetime demand_mwh
0  Jan 1, 2020 12:00 AM  43,500.50
1   Jan 1, 2020 1:00 AM  42,598.75
2   Jan 1, 2020 2:00 AM  41,463.75
                    datetime demand_mwh
53997   Feb 27, 2026 9:00 PM  54,669.00
53998  Feb 27, 2026 10:00 PM  51,251.04
53999  Feb 27, 2026 11:00 PM  48,006.98


In [13]:
# Stack df1 and df2 on top of each other
consumption = pd.concat([df1, df2], ignore_index=True)

# Remove any duplicate rows just in case
consumption = consumption.drop_duplicates(subset=['datetime'])

# Sort by time
consumption = consumption.sort_values('datetime').reset_index(drop=True)

# Check the result
print("Total rows:", consumption.shape)
print("First entry:", consumption['datetime'].iloc[0])
print("Last entry:", consumption['datetime'].iloc[-1])
print(consumption.head(3))

Total rows: (97813, 2)
First entry: Apr 1, 2015 10:00 AM
Last entry: Sep 9, 2025 9:00 PM
               datetime demand_mwh
0  Apr 1, 2015 10:00 AM  72,256.75
1  Apr 1, 2015 10:00 PM  61,074.00
2  Apr 1, 2015 11:00 AM  73,789.25


In [14]:
# Stack both files
consumption = pd.concat([df1, df2], ignore_index=True)

# Remove duplicates
consumption = consumption.drop_duplicates(subset=['datetime'])

# Convert datetime column from text to real datetime
consumption['datetime'] = pd.to_datetime(consumption['datetime'])

# NOW sort correctly by actual date
consumption = consumption.sort_values('datetime').reset_index(drop=True)

# Check
print("Total rows:", consumption.shape)
print("First entry:", consumption['datetime'].iloc[0])
print("Last entry:", consumption['datetime'].iloc[-1])
print(consumption.head(3))

C:\Users\acer\AppData\Local\Temp\ipykernel_21624\2149115513.py:8: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  consumption['datetime'] = pd.to_datetime(consumption['datetime'])


Total rows: (97813, 2)
First entry: 2015-01-01 00:00:00
Last entry: 2026-02-27 23:00:00
             datetime demand_mwh
0 2015-01-01 00:00:00  44,600.25
1 2015-01-01 01:00:00  43,454.75
2 2015-01-01 02:00:00  41,963.25


Now we will load Energy Generation data

In [15]:
# Load the three generation files
gen1 = pd.read_csv(
    "data/raw/smard_electricity/Actual_generation_201501010000_202101010000_Hour.csv",
    sep=None,
    engine='python',
    encoding='utf-8-sig'
)

print("Shape:", gen1.shape)
print("Columns:", list(gen1.columns))
print(gen1.head(3))

Shape: (52608, 14)
Columns: ['Start date', 'End date', 'Biomass [MWh] Calculated resolutions', 'Hydropower [MWh] Calculated resolutions', 'Wind offshore [MWh] Calculated resolutions', 'Wind onshore [MWh] Calculated resolutions', 'Photovoltaics [MWh] Calculated resolutions', 'Other renewable [MWh] Calculated resolutions', 'Nuclear [MWh] Calculated resolutions', 'Lignite [MWh] Calculated resolutions', 'Hard coal [MWh] Calculated resolutions', 'Fossil gas [MWh] Calculated resolutions', 'Hydro pumped storage [MWh] Calculated resolutions', 'Other conventional [MWh] Calculated resolutions']
             Start date             End date  \
0  Jan 1, 2015 12:00 AM  Jan 1, 2015 1:00 AM   
1   Jan 1, 2015 1:00 AM  Jan 1, 2015 2:00 AM   
2   Jan 1, 2015 2:00 AM  Jan 1, 2015 3:00 AM   

  Biomass [MWh] Calculated resolutions  \
0                             4,024.25   
1                             3,982.75   
2                             4,019.50   

  Hydropower [MWh] Calculated resolutions  \
0

In [16]:
# Print all column names one by one
for col in gen1.columns:
    print(col)

Start date
End date
Biomass [MWh] Calculated resolutions
Hydropower [MWh] Calculated resolutions
Wind offshore [MWh] Calculated resolutions
Wind onshore [MWh] Calculated resolutions
Photovoltaics [MWh] Calculated resolutions
Other renewable [MWh] Calculated resolutions
Nuclear [MWh] Calculated resolutions
Lignite [MWh] Calculated resolutions
Hard coal [MWh] Calculated resolutions
Fossil gas [MWh] Calculated resolutions
Hydro pumped storage [MWh] Calculated resolutions
Other conventional [MWh] Calculated resolutions


In [17]:
# Keep only the columns relevant to our research
gen1_clean = gen1[[
    'Start date',
    'Wind offshore [MWh] Calculated resolutions',
    'Wind onshore [MWh] Calculated resolutions',
    'Photovoltaics [MWh] Calculated resolutions',
    'Fossil gas [MWh] Calculated resolutions'
]]

# Rename columns to simple names
gen1_clean.columns = ['datetime', 'wind_offshore_mwh', 'wind_onshore_mwh', 'solar_mwh', 'gas_mwh']

# Convert datetime
gen1_clean['datetime'] = pd.to_datetime(gen1_clean['datetime'])

print("Shape:", gen1_clean.shape)
print(gen1_clean.head(3))

C:\Users\acer\AppData\Local\Temp\ipykernel_21624\3504888163.py:14: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  gen1_clean['datetime'] = pd.to_datetime(gen1_clean['datetime'])


Shape: (52608, 5)
             datetime wind_offshore_mwh wind_onshore_mwh solar_mwh   gas_mwh
0 2015-01-01 00:00:00            516.50         8,128.00      0.00  1,226.25
1 2015-01-01 01:00:00            516.25         8,297.50      0.00    870.75
2 2015-01-01 02:00:00            514.00         8,540.00      0.00    809.50


C:\Users\acer\AppData\Local\Temp\ipykernel_21624\3504888163.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gen1_clean['datetime'] = pd.to_datetime(gen1_clean['datetime'])


In [18]:
# Load all 3 generation files properly
gen_files = [
    "data/raw/smard_electricity/Actual_generation_201501010000_202101010000_Hour.csv",
    "data/raw/smard_electricity/Actual_generation_202012310000_202601010000_Hour.csv",
    "data/raw/smard_electricity/Actual_generation_202512310000_202602270000_Hour.csv"
]

# Columns we want from each file
cols_to_keep = [
    'Start date',
    'Wind offshore [MWh] Calculated resolutions',
    'Wind onshore [MWh] Calculated resolutions',
    'Photovoltaics [MWh] Calculated resolutions',
    'Fossil gas [MWh] Calculated resolutions'
]

# Load each file and store in a list
gen_list = []

for file in gen_files:
    df = pd.read_csv(file, sep=None, engine='python', encoding='utf-8-sig')
    df = df[cols_to_keep].copy()   # .copy() fixes the warning
    df.columns = ['datetime', 'wind_offshore_mwh', 'wind_onshore_mwh', 'solar_mwh', 'gas_mwh']
    df['datetime'] = pd.to_datetime(df['datetime'], dayfirst=False)  # fixes date warning
    gen_list.append(df)
    print(f"Loaded: {file.split('/')[-1]} → {df.shape}")

# Combine all 3 into one
generation = pd.concat(gen_list, ignore_index=True)
generation = generation.drop_duplicates(subset=['datetime'])
generation = generation.sort_values('datetime').reset_index(drop=True)

print("\nFinal generation dataset:")
print("Total rows:", generation.shape)
print("First entry:", generation['datetime'].iloc[0])
print("Last entry:", generation['datetime'].iloc[-1])
print(generation.head(3))

C:\Users\acer\AppData\Local\Temp\ipykernel_21624\600964219.py:24: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['datetime'] = pd.to_datetime(df['datetime'], dayfirst=False)  # fixes date warning


Loaded: Actual_generation_201501010000_202101010000_Hour.csv → (52608, 5)


C:\Users\acer\AppData\Local\Temp\ipykernel_21624\600964219.py:24: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['datetime'] = pd.to_datetime(df['datetime'], dayfirst=False)  # fixes date warning


Loaded: Actual_generation_202012310000_202601010000_Hour.csv → (43848, 5)
Loaded: Actual_generation_202512310000_202602270000_Hour.csv → (1392, 5)


C:\Users\acer\AppData\Local\Temp\ipykernel_21624\600964219.py:24: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['datetime'] = pd.to_datetime(df['datetime'], dayfirst=False)  # fixes date warning



Final generation dataset:
Total rows: (97789, 5)
First entry: 2015-01-01 00:00:00
Last entry: 2026-02-26 23:00:00
             datetime wind_offshore_mwh wind_onshore_mwh solar_mwh   gas_mwh
0 2015-01-01 00:00:00            516.50         8,128.00      0.00  1,226.25
1 2015-01-01 01:00:00            516.25         8,297.50      0.00    870.75
2 2015-01-01 02:00:00            514.00         8,540.00      0.00    809.50


In [19]:
# Merge consumption and generation on datetime
energy = pd.merge(consumption, generation, on='datetime', how='inner')

# Check the result
print("Shape:", energy.shape)
print("First entry:", energy['datetime'].iloc[0])
print("Last entry:", energy['datetime'].iloc[-1])
print(energy.head(3))

Shape: (97789, 6)
First entry: 2015-01-01 00:00:00
Last entry: 2026-02-26 23:00:00
             datetime demand_mwh wind_offshore_mwh wind_onshore_mwh solar_mwh  \
0 2015-01-01 00:00:00  44,600.25            516.50         8,128.00      0.00   
1 2015-01-01 01:00:00  43,454.75            516.25         8,297.50      0.00   
2 2015-01-01 02:00:00  41,963.25            514.00         8,540.00      0.00   

    gas_mwh  
0  1,226.25  
1    870.75  
2    809.50  


In [20]:
# Save energy dataset so other notebooks can use it
os.makedirs("data/processed", exist_ok=True)
energy.to_csv("data/processed/energy_clean.csv", index=False)
print("Energy data saved!")
print("Shape:", energy.shape)

Energy data saved!
Shape: (97789, 6)
